# Agentic RAG Demo

This notebook demonstrates the four user-facing execution paths in the project:

- `llm_direct` for conceptual questions
- `vectorstore` for local documentation lookup
- `web_search` for current information
- multi-hop planning for questions that need more than one source or sub-question

For each case, the notebook shows the selected route, tier, plan type, answer, and source citations.


## Notes

- Run this notebook from the project environment with `.env` configured.
- Build the local index first with `python -m src.ingestion.ingestion_test`.
- The `llm_direct` branch does not retrieve from the vector store or web search, so it has no citations by design.
- Current-information questions depend on Brave Search and external network access.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import Markdown, display

from src.api import run_adaptive_query


In [ ]:
DEMO_CASES = {
    'llm_direct': 'What is a professional certification?',
    'vectorstore': 'What does the Claude certification exam guide cover?',
    'web_search': 'As of today, is Claude 101 currently listed on the Anthropic Courses site?',
    'multi_hop': 'Using the local certification exam guide PDF and the current Anthropic Courses site, which currently listed course is most directly aligned with MCP preparation?',
}

demo_results = {}


In [ ]:
def run_demo_case(name, *, use_cache=False):
    question = DEMO_CASES[name]
    result = run_adaptive_query(question=question, use_cache=use_cache)
    demo_results[name] = result
    return result


## 1. `llm_direct` branch

This case shows a conceptual question that should be answered directly without retrieval.


In [ ]:
run_demo_case('llm_direct', use_cache=False)


## 2. `vectorstore` branch

This case shows local-document retrieval from the indexed certification corpus.


In [ ]:
run_demo_case('vectorstore', use_cache=False)


## 3. `web_search` branch

This case shows a current-information question that should route to web search and return live citations.


In [ ]:
run_demo_case('web_search', use_cache=False)


## 4. Multi-hop branch

This case shows decomposition across multiple sources and a merged final answer.


In [ ]:
run_demo_case('multi_hop', use_cache=False)


## Summary Table

This summary makes it easy to compare route selection, tiering, and citation behavior across the demo cases.


In [ ]:
def _escape_pipes(text):
    return str(text).replace('|', '\\|').replace('\n', '<br>')


def _citation_text(citations):
    if not citations:
        return 'none'
    return '<br>'.join(_escape_pipes(citation) for citation in citations)


rows = [
    '| Case | Question | Tier | Plan Type | Final Route | Route Reason | Citation Count | Citations |',
    '| --- | --- | --- | --- | --- | --- | ---: | --- |',
]

for name in ['llm_direct', 'vectorstore', 'web_search', 'multi_hop']:
    result = demo_results.get(name)
    if not result:
        continue
    rows.append(
        f"| {name} | {_escape_pipes(DEMO_CASES[name])} | {_escape_pipes(result.get('tier_label', 'n/a'))} | {_escape_pipes(result.get('plan_type', 'n/a'))} | {_escape_pipes(result.get('final_route', 'n/a'))} | {_escape_pipes(result.get('plan_reason', 'n/a'))} | {len(result.get('citations', []))} | {_citation_text(result.get('citations', []))} |"
    )

display(Markdown('\n'.join(rows)))
